In [57]:
t = ([1], 2)
t[0].append(100)
t[0][0] = 9
print(t)
t[0].remove(9)
print(t)
t[0].remove(100)
print(t)

([9, 100], 2)
([100], 2)
([], 2)


## 4.8 Object Protocols and Data Abstraction

In [6]:
def compute_cost(unit_price: float, num_units: int) -> float:
    return unit_price * num_units

print(compute_cost(10, 5))
print(compute_cost(3.5, 4))
print(compute_cost("test ", 10))

50
14.0
test test test test test test test test test test 


## 4.11 Comparison Protocol

In [ ]:
# `is` operator can't be redefined
a = [1, 2, 3]
b = a
print(a is b)

c = [1, 2, 3]
print(a is c)

True
False


In [9]:
class Money:
    def __init__(self, v): self.v = v
    def __add__(self, other):
        if isinstance(other, Money): return Money(self.v + other.v)
        if isinstance(other, int):   return Money(self.v + other)
        return NotImplemented
    __radd__ = __add__   # sčítání je komutativní

print(5 + Money(3))                      # int.__add__ vrátí NotImplemented, pak Money.__radd__(5)
print(sum([Money(1), Money(2)]))         # sum začíná od 0, takže první krok je 0 + Money(1)


In [ ]:
a = 34
n = a.numerator # get numerator (an attribute)
print(n)

b = [1, 2, 3]
b.append(7)
print(b)

# writing a + 10 executes a method a.__add__(10)
c = a + 10
print(c)

d = b + [4, 5]
print(d)

34
[1, 2, 3, 7]
44
[1, 2, 3, 7, 4, 5]


In [12]:
# compare two objects
def compare(a, b):
    if a is b:
        print("same object")
    if a == b:
        print("same value")
    if type(a) is type(b):
        print("same type")

a = [1, 2, 3]
b = [1, 2, 3]
compare(a, b)
print("---")
compare(a, [4, 5, 6])

same value
same type
---
same type


In [19]:
# 1) Add a value to a list
items = []
items.append(5)
print("list after append:", items)

# 2) Function: remove every copy of a value
def removeall(items, item):
    return [i for i in items if i != item]

print("remove 2:", removeall([1, 2, 3, 2], 2))

# 3) Same thing as a method on our own list class
class mylist(list):
    def removeall(self, val):
        return [i for i in self if i != val]

items = mylist([5, 8, 2, 7, 2, 13, 9])
print("mylist without 2:", items.removeall(2))

# 4) Biggest number
print("max:", max(items))

list after append: [5]
remove 2: [1, 3]
mylist without 2: [5, 8, 7, 13, 9]
max: 13


In [20]:
# 5) Why it doesn't work with deque
from collections import deque

d = deque([5, 8, 2, 7])

# deque is NOT a list or tuple, so this check is False
print("is list or tuple?", isinstance(d, (list, tuple)))

if isinstance(d, (list, tuple)):
    print("max:", max(d))
else:
    print("skipped, even though max(d) works:", max(d))

# deque has no removeall method (only mylist has it)
print("has removeall?", hasattr(d, "removeall"))

# the plain function works, because it just loops over the items
print("function removeall:", removeall(d, 2))

is list or tuple? False
skipped, even though max(d) works: 8
has removeall? False
function removeall: [5, 8, 7]


In [18]:
items = list()

if isinstance(items, list):
    items.append(items)

def removeall(items: list, item) -> list:
    return [i for i in items if i != item]

class mylist(list):
    def removeall(self, val):
        return [i for i in self if i != val]

# example
items = mylist([5, 8, 2, 7, 2, 13, 9])
x = items.removeall(2)
print(x)


if isinstance(items, (list, tuple)):
    maxval = max(items)

[5, 8, 7, 13, 9]


## Reference Counting and Garbage Collection

In [ ]:
# since Python 3.12 small numbers are "immortal"
# integers from -5 to 256 are cached at startup and are reused everywhere
# so every 256 is one shared object, from 257 up you get normal objects
import sys
import gc

print("256:", sys.getrefcount(256))              # immortal
print("256 is 256?", int("256") is int("256"))   # True, one shared object
print("257 is 257?", int("257") is int("257"))   # False, two new objects

a = 257
print("start:", sys.getrefcount(a))   # +1 extra for the function call itself

b = a
print("after b = a:", sys.getrefcount(a))

c = []
c.append(b)
print("after c.append(b):", sys.getrefcount(a))

del b
print("after del b:", sys.getrefcount(a))

c[0] = 2.0
print("after c[0] = 2.0:", sys.getrefcount(a))

# Circular dependency
x = {}
y = {}
x["y"] = y             # x points to y
y["x"] = x             # y points to x
del x
del y
# they won't get cleaned until cycle detector executes
print("collected by gc:", gc.collect())   # garbage collector cleans them up

256: 3221225472
256 is 256? True
257 is 257? False
start: 3
after b = a: 4
after c.append(b): 5
after del b: 4
after c[0] = 2.0: 3
collected by gc: 2


## References and Copies

In [46]:
import copy

print("Reference:")
a = [1, 2, 3, 4]
b = a # b is reference to a
print("b is a?", b is a)          # True
b[2] = -100                       # change b
print("a also changed:", a)       # [1, 2, -100, 4]

# 2) Shallow copy: new outer list, but inner items are shared
print("\nShallow copy:")
a = [1, 2, [3, 4]]
b = list(a)
print(b)
print("b is a?", b is a)          # False, different lists
b.append(100)
print("b:", b)                    # [1, 2, [3, 4], 100]
print("a unchanged:", a)          # [1, 2, [3, 4]]
b[2][0] = -100                    # change the inner list
print("b:", b)                    # [1, 2, [-100, 4], 100]
print("a changed inside:", a)     # [1, 2, [-100, 4]]
print("inner list shared?", a[2] is b[2])   # True

# 3) Deep copy: everything is copied, nothing shared
print("\nDeep copy:")
a = [1, 2, [3, 4]]
b = copy.deepcopy(a)
b[2][0] = -100
print("b:", b)                    # [1, 2, [-100, 4]]
print("a unchanged:", a)          # [1, 2, [3, 4]]
print("inner list shared?", a[2] is b[2])   # False

Reference:
b is a? True
a also changed: [1, 2, -100, 4]

Shallow copy:
[1, 2, [3, 4]]
b is a? False
b: [1, 2, [3, 4], 100]
a unchanged: [1, 2, [3, 4]]
b: [1, 2, [-100, 4], 100]
a changed inside: [1, 2, [-100, 4]]
inner list shared? True

Deep copy:
b: [1, 2, [-100, 4]]
a unchanged: [1, 2, [3, 4]]
inner list shared? False
